In [1]:
import pandas as pd

In [2]:
data = {
    "customer_id": ["C001", "C002", "C001", "C002", "C001", "C002"],
    "month": [
        "2026-01-01", "2026-01-01",
        "2026-02-01", "2026-02-01",
        "2026-03-01", "2026-03-01"
    ],
    "mrr": [1000, 2000, 1200, 1800, 1500, 2100]
}

df = pd.DataFrame(data)
df["month"] = pd.to_datetime(df["month"])

In [3]:
df.head()

,customer_id,month,mrr
0,C001,2026-01-01,1000
1,C002,2026-01-01,2000
2,C001,2026-02-01,1200
3,C002,2026-02-01,1800
4,C001,2026-03-01,1500


In [5]:
df = df.sort_values('month')

In [6]:
df

,customer_id,month,mrr
0,C001,2026-01-01,1000
1,C002,2026-01-01,2000
2,C001,2026-02-01,1200
3,C002,2026-02-01,1800
4,C001,2026-03-01,1500
5,C002,2026-03-01,2100


Create:

- previous_mrr

- containing each customer's previous month's MRR.

- The first record for each customer should naturally have no previous value.

In [9]:
df['prev_month_mrr'] = df.groupby('customer_id')['mrr'].shift(1)

In [10]:
df

,customer_id,month,mrr,prev_month_mrr
0,C001,2026-01-01,1000,NaN
1,C002,2026-01-01,2000,NaN
2,C001,2026-02-01,1200,1000.0
3,C002,2026-02-01,1800,2000.0
4,C001,2026-03-01,1500,1200.0
5,C002,2026-03-01,2100,1800.0


Create:

- mrr_change

- Expected business meaning:

    - current MRR - previous MRR

In [12]:
df['mrr_change'] = df.groupby('customer_id')['mrr'].diff()

In [13]:
df

,customer_id,month,mrr,prev_month_mrr,mrr_change
0,C001,2026-01-01,1000,NaN,NaN
1,C002,2026-01-01,2000,NaN,NaN
2,C001,2026-02-01,1200,1000.0,200.0
3,C002,2026-02-01,1800,2000.0,-200.0
4,C001,2026-03-01,1500,1200.0,300.0
5,C002,2026-03-01,2100,1800.0,300.0


Create:

- mrr_change_pct

- Represent the percentage as:

    - 20.0
    - -10.0
    - 25.0

rather than decimals such as 0.20.

In [17]:
df['mrr_change_pct'] = df.groupby('customer_id')['mrr'].pct_change() * 100

In [18]:
df

,customer_id,month,mrr,prev_month_mrr,mrr_change,mrr_change_pct
0,C001,2026-01-01,1000,NaN,NaN,NaN
1,C002,2026-01-01,2000,NaN,NaN,NaN
2,C001,2026-02-01,1200,1000.0,200.0,20.000000
3,C002,2026-02-01,1800,2000.0,-200.0,-10.000000
4,C001,2026-03-01,1500,1200.0,300.0,25.000000
5,C002,2026-03-01,2100,1800.0,300.0,16.666667


- Assume each row represents that customer's monthly billed revenue.

- Create:

    - lifetime_revenue

- which accumulates independently for each customer.

In [20]:
df['lifetime_revenue'] = df.groupby('customer_id')['mrr'].cumsum()

In [21]:
df

,customer_id,month,mrr,prev_month_mrr,mrr_change,mrr_change_pct,lifetime_revenue
0,C001,2026-01-01,1000,NaN,NaN,NaN,1000
1,C002,2026-01-01,2000,NaN,NaN,NaN,2000
2,C001,2026-02-01,1200,1000.0,200.0,20.000000,2200
3,C002,2026-02-01,1800,2000.0,-200.0,-10.000000,3800
4,C001,2026-03-01,1500,1200.0,300.0,25.000000,3700
5,C002,2026-03-01,2100,1800.0,300.0,16.666667,5900


Create:

subscription_month_number

such that each customer's history becomes:

1
2
3
...

In [24]:
df['subscription_month_number'] = df.groupby('customer_id')['mrr'].cumcount() + 1

In [25]:
df

,customer_id,month,mrr,prev_month_mrr,mrr_change,mrr_change_pct,lifetime_revenue,subscription_month_number
0,C001,2026-01-01,1000,NaN,NaN,NaN,1000,1
1,C002,2026-01-01,2000,NaN,NaN,NaN,2000,1
2,C001,2026-02-01,1200,1000.0,200.0,20.000000,2200,2
3,C002,2026-02-01,1800,2000.0,-200.0,-10.000000,3800,2
4,C001,2026-03-01,1500,1200.0,300.0,25.000000,3700,3
5,C002,2026-03-01,2100,1800.0,300.0,16.666667,5900,3
